# Train a CNN+FNN to follow the sheep

The dataset from **`generate_fox_vision_ds.ipynb`** pairs each fox view with many candidate
headings, labelled **1** (toward the sheep) or **0** (a wrong heading — sideways, oblique,
opposite, …). We train a **direction scorer**:

> **CNN**(terrain + entity view) ⊕ **FNN**(candidate heading) → *P(this heading is good)*.

Because the negatives now span the whole wrong-direction space, the model must learn that
**only headings pointing at the sheep are good** — sideways and oblique are rejected too, not
just the exact opposite.

Then we **recover the movement direction**: score a fan of 72 candidate headings for a held-out
view and take the best one. If that recovered heading points at the sheep, the model has
learned to follow it — from vision alone.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

_here = Path.cwd()
REPO = next((c for c in [_here, *_here.parents]
             if (c / "config.py").exists() and (c / "sim").is_dir()), _here)
DATA_PATH = REPO / "notebooks" / "fox_vision_dataset.csv"

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| data:", DATA_PATH)

device: cpu | data: c:\Users\afree\Desktop\ecosystem\notebooks\fox_vision_dataset.csv


In [2]:
torch.cuda.is_available()

False

In [ ]:
df = pd.read_csv(DATA_PATH)
WIN = int(round(int(df.columns.str.startswith("t_").sum()) ** 0.5))
t_cols = [f"t_{i}" for i in range(WIN*WIN)]
e_cols = [f"e_{i}" for i in range(WIN*WIN)]

terr = df[t_cols].to_numpy(np.float32).reshape(-1, WIN, WIN)
enti = df[e_cols].to_numpy(np.float32).reshape(-1, WIN, WIN)
X = np.stack([terr, enti], axis=1)               # (N, 2, WIN, WIN)
dvec  = df[["dx", "dy"]].to_numpy(np.float32)     # the candidate heading (model input)
label = df["label"].to_numpy(np.float32)          # 1 = good heading, 0 = wrong
scen  = df["scenario_id"].to_numpy(np.int64)
sheep = df[["sheep_dx", "sheep_dy"]].to_numpy(np.float32)
print("X:", X.shape, "| rows:", len(df), "| scenarios:", df.scenario_id.nunique(),
      "| positives:", int(label.sum()), "negatives:", int((1-label).sum()))

In [ ]:
# Split by SCENARIO so a scene's rows (identical pixels) never straddle train/val.
uniq = np.unique(scen)
rng = np.random.default_rng(0); rng.shuffle(uniq)
n_val = max(1, int(0.2 * len(uniq)))
val_scen = set(uniq[:n_val].tolist())
is_val = np.array([s in val_scen for s in scen])

Xtr = torch.tensor(X[~is_val]);   dtr = torch.tensor(dvec[~is_val]); ltr = torch.tensor(label[~is_val])
Xva = torch.tensor(X[is_val]);    dva = torch.tensor(dvec[is_val]);  lva = torch.tensor(label[is_val])
print(f"train rows {len(Xtr)} | val rows {len(Xva)}")

In [ ]:
class FoxDirScorer(nn.Module):
    """CNN over the 2-channel view + FNN over a candidate heading -> P(heading is good)."""
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(6),                     # -> (32, 6, 6)
            nn.Flatten(),
        )
        self.dir_fc = nn.Sequential(nn.Linear(2, 32), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(32*6*6 + 32, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),                            # logit
        )
    def forward(self, grid, d):
        f = self.cnn(grid)
        return self.head(torch.cat([f, self.dir_fc(d)], dim=1)).squeeze(1)

model = FoxDirScorer().to(device)
print(model)
print("trainable params:", sum(p.numel() for p in model.parameters()))

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
# negatives outnumber positives 2:1 (4 wrong headings vs 2 good per scene); weight the
# positive class so the optimizer can't just park at "predict every heading wrong".
pos_weight = torch.tensor([(ltr == 0).sum() / max((ltr == 1).sum(), 1)], device=device)
lossfn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
Xtr_d, dtr_d, ltr_d = Xtr.to(device), dtr.to(device), ltr.to(device)
Xva_d, dva_d, lva_d = Xva.to(device), dva.to(device), lva.to(device)

EPOCHS, BATCH = 40, 128
n = len(Xtr_d)
for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n, device=device)
    tot = 0.0
    for i in range(0, n, BATCH):
        b = perm[i:i+BATCH]
        loss = lossfn(model(Xtr_d[b], dtr_d[b]), ltr_d[b])
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(b)
    if ep == 1 or ep % 5 == 0:
        model.eval()
        with torch.no_grad():
            pv = torch.sigmoid(model(Xva_d, dva_d))
            acc = ((pv > 0.5).float() == lva_d).float().mean().item()
        print(f"epoch {ep:2d}  train_loss {tot/n:.4f}   val acc {acc*100:5.1f}%")

## The proof — recover the heading and beat every wrong direction

For each held-out scene we score a **fan of 72 candidate headings** and pick the best. The
recovered heading should point at the sheep. We also plot the score as a function of angle
away from the sheep: it should peak at 0° and fall off through sideways (±90°) to the
opposite (180°).

In [ ]:
model.eval()

# one representative row per validation scene (rows in a scene share the same view)
val_rows = np.where(is_val)[0]
rep = {}
for i in val_rows:
    rep.setdefault(int(scen[i]), i)
rep_idx = np.array(list(rep.values()))

FAN = 72
fan_ang = np.linspace(0, 2*np.pi, FAN, endpoint=False)
fan_dirs = np.stack([np.cos(fan_ang), np.sin(fan_ang)], axis=1).astype(np.float32)
fan_t = torch.tensor(fan_dirs, device=device)

rec_cos, ang_err = [], []
curve = np.zeros(FAN)                                 # mean score vs offset-from-true angle
with torch.no_grad():
    for gi in rep_idx:
        grid = torch.tensor(X[gi:gi+1], device=device).repeat(FAN, 1, 1, 1)
        sc = torch.sigmoid(model(grid, fan_t)).cpu().numpy()
        true_ang = np.arctan2(sheep[gi, 1], sheep[gi, 0])
        # recovered heading = best-scored candidate
        best = fan_dirs[int(sc.argmax())]
        tw = np.array([np.cos(true_ang), np.sin(true_ang)], np.float32)
        rec_cos.append(float(best @ tw))
        ang_err.append(np.degrees(np.arccos(np.clip(best @ tw, -1, 1))))
        # align this scene's scores to offset-from-true, then accumulate
        shift = int(round((true_ang % (2*np.pi)) / (2*np.pi) * FAN))
        curve += np.roll(sc, -shift)
curve /= len(rep_idx)
rec_cos = np.array(rec_cos); ang_err = np.array(ang_err)

print(f"held-out scenes: {len(rep_idx)}")
print(f"recovered-heading cosine to sheep:  mean {rec_cos.mean():+.3f}")
print(f"recovered within 20 deg of sheep:   {np.mean(ang_err <= 20)*100:.1f}%")
print(f"median heading error:               {np.median(ang_err):.1f} deg")

offsets = np.degrees(((fan_ang + np.pi) % (2*np.pi)) - np.pi)
o = np.argsort(offsets)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(offsets[o], curve[o], "-o", ms=3)
ax[0].axvline(0, color="green", ls="--", lw=1, label="toward sheep")
ax[0].axvline(90, color="orange", ls=":", lw=1); ax[0].axvline(-90, color="orange", ls=":", lw=1, label="sideways")
ax[0].axvline(180, color="red", ls=":", lw=1); ax[0].axvline(-180, color="red", ls=":", lw=1, label="opposite")
ax[0].set_xlabel("heading offset from sheep (deg)"); ax[0].set_ylabel("mean score P(good)")
ax[0].set_title("Score peaks toward the sheep, falls off sideways/opposite"); ax[0].legend(fontsize=8)
ax[1].hist(ang_err, bins=30, color="teal")
ax[1].set_xlabel("recovered-heading error (deg)"); ax[1].set_ylabel("scenes")
ax[1].set_title(f"Recovered heading error (median {np.median(ang_err):.1f}°)")
plt.tight_layout(); plt.show()

In [ ]:
# Qualitative: recovered heading (red) vs true toward-sheep (green) on a few held-out views.
pick = np.random.default_rng(1).choice(rep_idx, size=4, replace=False)
c = WIN // 2
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
with torch.no_grad():
    for a, gi in zip(axes, pick):
        grid = torch.tensor(X[gi:gi+1], device=device).repeat(FAN, 1, 1, 1)
        sc = torch.sigmoid(model(grid, fan_t)).cpu().numpy()
        best = fan_dirs[int(sc.argmax())]
        tw = sheep[gi] / (np.linalg.norm(sheep[gi]) + 1e-9)
        a.imshow(X[gi, 1], origin="upper", cmap="magma")     # entity channel
        a.plot(c, c, "co", ms=8, mec="k")
        a.arrow(c, c, tw[0]*5, tw[1]*5, color="lime", width=0.25, head_width=1.2,
                length_includes_head=True, label="true")
        a.arrow(c, c, best[0]*5, best[1]*5, color="red", width=0.16, head_width=1.0,
                length_includes_head=True, label="recovered")
        a.set_title(f"cos={float(best @ tw):.2f}"); a.legend(loc="upper right", fontsize=7)
fig.suptitle("Entity channel: recovered (red) vs true (green) heading to the sheep")
plt.tight_layout(); plt.show()

## Conclusion

With negatives that span the **whole** wrong-direction space — sideways, oblique, and
opposite — the scorer learns that *only* headings pointing at the sheep are good. Scoring a
fan of candidate headings and taking the best **recovers the correct movement direction**
from the fox's terrain + entity view alone, with a small angular error.

This confirms the hypothesis: **a learned CNN+FNN can follow (and thus chase / catch) an
entity from egocentric perception alone**, and it plugs in behind the same
`Brain.decide(obs) → act` contract the `RuleBrain` uses (a neural fox would score its
candidate headings each tick and move along the best one).